<style>
.jp-Notebook .jp-MarkdownCell,
.text_cell_render {
    font-size: 14px;
    line-height: 1.38;
}
.jp-CodeCell .cm-editor,
.jp-CodeCell pre,
.code_cell .CodeMirror {
    font-size: 13px !important;
    line-height: 1.30 !important;
}
</style>

# ECON 422: Macroeconomics & Machine Learning
## Multi-agent control tower

**Run the first code cell, then edit and run the second code cell.** The interactive control tower appears as a link in local Jupyter and as an embedded panel in Google Colab. This notebook includes the required project scripts and dashboard. Standard Python packages must be installed in your Jupyter kernel.

## Step 1. Prepare the dashboard
Run the next cell once. It creates a versioned project folder beside your notebook and starts the dashboard server. **Local Jupyter** opens it through a localhost link. **Google Colab** keeps the server inside the remote runtime and displays it through Colab's authenticated kernel-port proxy in Step 2.


In [ ]:
# STEP 1: Prepare the included project files and start the local dashboard.
# No separate project-file downloads are needed.
from pathlib import Path
import base64, gzip, hashlib, importlib, json, re, subprocess, sys, time
from IPython.display import display, Markdown

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

_BUNDLED_FILES = '...'
_payload = gzip.decompress(base64.b64decode(_BUNDLED_FILES))
_assets = json.loads(_payload)
APP_DIR = Path.cwd() / ("control_tower_app_" + hashlib.sha256(_payload).hexdigest()[:12])
APP_DIR.mkdir(exist_ok=True)
for _name, _text in _assets.items():
    _destination = APP_DIR / _name
    if _destination.exists() and _destination.read_text(encoding="utf-8") != _text:
        raise RuntimeError(f"Bundled file was edited: {_destination}. Rename that app folder to preserve your edits, then rerun this cell.")
    _destination.write_text(_text, encoding="utf-8")
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))
# Discard an older imported project module so all cells use this bundle.
for _module in ("simulation_support", "ml_analyst", "model_evaluation_analyst"):
    _loaded = sys.modules.get(_module)
    if _loaded is not None and Path(getattr(_loaded, "__file__", "")).parent != APP_DIR:
        del sys.modules[_module]
importlib.invalidate_caches()

_missing = [name for name in ("numpy", "pandas", "sklearn", "joblib", "matplotlib", "shap")
            if importlib.util.find_spec(name) is None]
if _missing:
    print("Missing Python packages:", ", ".join(_missing))
    print("Run this in a temporary notebook cell, then rerun Step 1:")
    print(f'%pip install -r "{APP_DIR / "requirements.txt"}"')
    raise RuntimeError("Install the listed Python packages before continuing.")

if "control_tower_process" not in globals() or control_tower_process.poll() is not None:
    dashboard_log_path = APP_DIR / "dashboard_server.log"
    with dashboard_log_path.open("w", encoding="utf-8") as dashboard_log:
        _dashboard_cmd = [sys.executable, str(APP_DIR / "dashboard.py"), "--no-browser"]
        if IN_COLAB:
            _dashboard_cmd.append("--allow-colab-proxy")
        control_tower_process = subprocess.Popen(
            _dashboard_cmd,
            cwd=APP_DIR, stdout=dashboard_log, stderr=subprocess.STDOUT,
        )
    for _ in range(100):
        log_text = dashboard_log_path.read_text(encoding="utf-8")
        if control_tower_process.poll() is not None:
            raise RuntimeError(log_text or "Dashboard exited before starting.")
        match = re.search(r"http://127\.0\.0\.1:\d+", log_text)
        if match:
            dashboard_url = match.group(0)
            dashboard_port = int(dashboard_url.rsplit(":", 1)[1])
            break
        time.sleep(0.1)
    else:
        control_tower_process.terminate()
        raise RuntimeError(f"Startup timed out. Inspect {dashboard_log_path}.")
print("Step 1 complete. Edit and run the SECOND code cell next.")
print("The interactive control tower will appear in Step 2 (embedded in Colab; link in local Jupyter).")
print("Project files and browser outputs:", APP_DIR)
# To stop this dashboard: control_tower_process.terminate()


## Step 2. Type your DGP and open the interactive window
Edit the code inside `DGP_CODE` below, including `growth = (...)`, and run this cell. It generates a preview and provides **Open interactive window with my DGP**. Click that link, choose **Generate simulated data → My notebook DGP**, and click **Launch analysts**.

After an edit, rerun this cell and use its new link. Keep `df`, `FEATURES`, `TARGET`, `DATASET_LABEL`, and `TARGET_UNITS`. Set `USE_CUSTOM_DGP = False` for the original example. The browser regenerates data from this code when you launch; keep a fixed seed for reproducibility.

For browser analysis you can stop after this cell. The remaining cells provide an alternative notebook-based analysis.

To analyze your own CSV instead, choose **Upload a data file** in Configuration and specify the outcome and predictor columns. The window contains no duplicate DGP editor.

**Analysis settings** in the interactive window control cross-validation, the maximum number of SHAP observations, and the split/CV seed. They apply to either data source and do not change the DGP.

In [ ]:
# SECOND CODE CELL: TYPE YOUR DGP HERE.
# Edit only the settings and the code between the triple quotes.
USE_CUSTOM_DGP = True
DEFAULT_ROWS = 800
DEFAULT_SEED = 422

DGP_CODE = r"""

import numpy as np
import pandas as pd

# YOUR DGP: edit the distributions and growth equation below.
N = 800
SEED = 422
FEATURES = ['unemployment', 'credit_spread', 'inflation', 'investment_growth', 'confidence', 'policy_rate']
TARGET = "growth_next_year"
DATASET_LABEL = "SIMULATED: Nonlinear regional growth"
TARGET_UNITS = "percentage points"

rng = np.random.default_rng(SEED)
unemployment = rng.uniform(3, 11, N)
spread = rng.uniform(0.3, 4.5, N)
inflation = rng.normal(2.5, 1.2, N)
investment = rng.normal(3, 2, N)
confidence = 100 - 2 * unemployment - 3 * spread + rng.normal(0, 5, N)
policy_rate = 1 + 0.65 * inflation + rng.normal(0, 0.7, N)
growth = (3.5 - 0.25 * unemployment + 0.4 * investment
          - 0.18 * (inflation - 2)**2
          - 2.5 * ((unemployment > 6.5) & (spread > 2.3))
          + 0.015 * (confidence - 80) + rng.normal(0, 0.65, N))

df = pd.DataFrame(dict(
    unemployment=unemployment, credit_spread=spread,
    inflation=inflation, investment_growth=investment,
    confidence=confidence, policy_rate=policy_rate,
    growth_next_year=growth,
))
# Inject missing predictors; imputation stays inside the CV pipeline.
for column in ["confidence", "investment_growth"]:
    df.loc[rng.choice(N, size=max(1, N // 40), replace=False), column] = np.nan

# Required outputs: df, FEATURES, TARGET, DATASET_LABEL, TARGET_UNITS.
# Optional: df.to_csv("my_simulated_data.csv", index=False)
"""

# Apply your choice and prepare an optional dashboard link.
from pathlib import Path
from urllib.parse import quote
from IPython.display import display, Markdown
if "APP_DIR" not in globals():
    raise RuntimeError("Run the FIRST code cell once to prepare the included project files.")
from simulation_support import simulation_code, validate_code, validate_outputs

RUN_DGP_CODE = validate_code(
    DGP_CODE if USE_CUSTOM_DGP else
    simulation_code("nonlinear", DEFAULT_ROWS, DEFAULT_SEED)
)
_preview_scope = {"__name__": "__main__"}
exec(compile(RUN_DGP_CODE, "my_dgp.py", "exec"), _preview_scope)
preview_df, preview_metadata = validate_outputs(_preview_scope)
print(f"DGP works: generated {len(preview_df)} rows. Preview:")
display(preview_df.head())
if "dashboard_url" in globals() and control_tower_process.poll() is None:
    dgp_fragment = "#notebook-dgp=" + quote(RUN_DGP_CODE, safe="")
    if globals().get("IN_COLAB", False):
        from google.colab import output as colab_output
        if "dashboard_port" not in globals():
            dashboard_port = int(dashboard_url.rsplit(":", 1)[1])
        display(Markdown("### Interactive control tower"))
        colab_output.serve_kernel_port_as_iframe(
            dashboard_port,
            path="/" + dgp_fragment,
            height=900,
        )
        print("Colab is using its authenticated kernel-port proxy; do not open the 127.0.0.1 address directly.")
    else:
        dgp_url = dashboard_url.split("#", 1)[0] + dgp_fragment
        display(Markdown(f"[Open interactive window with my DGP]({dgp_url})"))
    print("In the dashboard, review the custom simulation and click Launch analysts.")
else:
    print("The dashboard is stopped. Rerun the first cell, then this cell for a new link.")
